In [ ]:

import os, glob
import numpy as np
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

import timm
from tqdm import tqdm


MODEL_PATH = r"E:\SFIAD_Project\models\sfiad_xception_fedavg_100pct1.pth"

DATASETS = {
    "FF++": r"E:\SFIAD_Project\ffpp_fused_npy\test",
    "CelebDF-v1": r"E:\SFIAD_Project\celebdf_v1new_fused_npy\test",
    "CelebDF-v2": r"E:\SFIAD_Project\celebdf_v2_fused_npy\test",
    "UADFV": r"E:\SFIAD_Project\uadfv_fused_npy\test",
    "DeepfakeTIMIT": r"E:\SFIAD_Project\timit_fused_npy\test"
}

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 32

class UniversalDataset(Dataset):
    def __init__(self, root):

        self.samples = []

        for cls in ["real", "fake"]:
            folder = os.path.join(root, cls)

            files = glob.glob(os.path.join(folder, "*.npy"))

            label = 0 if cls == "real" else 1

            for f in files:
                self.samples.append((f, label))

        print("Loaded Frames:", len(self.samples))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):

        path, label = self.samples[idx]

        x = np.load(path)

        if x.shape[0] != 4:
            x = np.transpose(x, (2,0,1))

        return torch.tensor(x, dtype=torch.float32), label, path



class AAMSoftmax(nn.Module):
    def __init__(self, feat_dim, n_classes=2):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(n_classes, feat_dim))

class Model(nn.Module):
    def __init__(self):
        super().__init__()

        self.backbone = timm.create_model(
            "xception",
            pretrained=False,
            num_classes=0,
            in_chans=4
        )

        feat_dim = self.backbone.num_features

        self.cls_loss = AAMSoftmax(feat_dim, 2)

    def forward(self, x):
        return self.backbone(x)


model = Model().to(DEVICE)

state = torch.load(MODEL_PATH, map_location=DEVICE)
model.load_state_dict(state, strict=False)

model.eval()

print("Model Loaded")

def evaluate_dataset(name, root):

    print("\n" + "="*60)
    print(name)
    print("="*60)

    dataset = UniversalDataset(root)

    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    frame_probs = []
    frame_gt = []

    video_scores = defaultdict(list)
    video_gt = {}

    with torch.no_grad():

        for x, y, paths in tqdm(loader):

            x = x.to(DEVICE)

            feats = model(x)

            logits = F.linear(
                F.normalize(feats.float()),
                F.normalize(model.cls_loss.weight.float())
            )

            probs = torch.softmax(logits, dim=1)[:,1].cpu().numpy()

            frame_probs.extend(probs)
            frame_gt.extend(y.numpy())

            for p, gt, path in zip(probs, y.numpy(), paths):

                vid = "_".join(
                    os.path.basename(path).split("_")[:-1]
                )

                video_scores[vid].append(p)
                video_gt[vid] = gt

    frame_probs = np.array(frame_probs)
    frame_gt = np.array(frame_gt)

    vid_probs = np.array([
        np.mean(video_scores[v]) for v in video_scores
    ])

    vid_gt = np.array([
        video_gt[v] for v in video_scores
    ])

    # Threshold
    preds = (vid_probs >= 0.5).astype(int)

    # Metrics
    acc = accuracy_score(vid_gt, preds)
    precision = precision_score(vid_gt, preds, zero_division=0)
    recall = recall_score(vid_gt, preds, zero_division=0)
    f1 = f1_score(vid_gt, preds, zero_division=0)

    try:
        auc = roc_auc_score(vid_gt, vid_probs)
        ap = average_precision_score(vid_gt, vid_probs)
    except:
        auc = np.nan
        ap = np.nan

    cm = confusion_matrix(vid_gt, preds)

    print("Videos      :", len(vid_gt))
    print("ACC         :", round(acc,4))
    print("Precision   :", round(precision,4))
    print("Recall      :", round(recall,4))
    print("F1 Score    :", round(f1,4))
    print("AUC         :", round(auc,4))
    print("AP          :", round(ap,4))
    print("Confusion Matrix:")
    print(cm)

for name, path in DATASETS.items():
    evaluate_dataset(name, path)